# Scraping de ciclos formativos de grado medio de Tartanga

En este notebook descargaremos la página de ciclos formativos de grado medio de Tartanga, visitaremos la ficha de cada ciclo y guardaremos la información en CSV y JSON.

Ejecuta las celdas en orden, de arriba abajo.

## 1. Instalar las librerías

Usaremos `requests` para descargar las páginas y `BeautifulSoup` para analizar el HTML.

In [2]:
%pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


## 2. Importar librerías

In [3]:
import csv
import json
from pathlib import Path

import requests
from bs4 import BeautifulSoup

print("Librerías listas")

Librerías listas


## 3. Descargar la página principal y localizar los ciclos

In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://www.tartanga.eus/formacion/ciclos-formativos-de-grado-medio-grados-d/"
respuesta = requests.get(url, timeout=30)
respuesta.raise_for_status()

soup = BeautifulSoup(respuesta.text, "html.parser")

enlaces_ciclos = []
for enlace in soup.select("a[href]"):
    href = enlace.get("href", "").rstrip("/")
    texto = " ".join(enlace.get_text(" ", strip=True).split())
    if href.startswith(url.rstrip("/") + "/") and texto:
        registro = {"nombre": texto, "url": href + "/"}
        if registro not in enlaces_ciclos:
            enlaces_ciclos.append(registro)

print("Página descargada:", respuesta.status_code)
print("Ciclos encontrados:", len(enlaces_ciclos))
for ciclo in enlaces_ciclos:
    print(ciclo)

Página descargada: 200
Ciclos encontrados: 2
{'nombre': 'Gestión administrativa', 'url': 'https://www.tartanga.eus/formacion/ciclos-formativos-de-grado-medio-grados-d/gestion-administrativa/'}
{'nombre': 'Instalaciones eléctricas y automáticas', 'url': 'https://www.tartanga.eus/formacion/ciclos-formativos-de-grado-medio-grados-d/instalaciones-electricas-y-automaticas/'}


## 4. Comprobar los ciclos encontrados

In [5]:
print("Título de la página:", soup.title.get_text(strip=True))
print("\nEnlaces de ciclos:")
for ciclo in enlaces_ciclos:
    print(f"- {ciclo['nombre']}: {ciclo['url']}")

Título de la página: Ciclos Formativos de Grado Medio – Grados D – CIFP Tartanga LHII

Enlaces de ciclos:
- Gestión administrativa: https://www.tartanga.eus/formacion/ciclos-formativos-de-grado-medio-grados-d/gestion-administrativa/
- Instalaciones eléctricas y automáticas: https://www.tartanga.eus/formacion/ciclos-formativos-de-grado-medio-grados-d/instalaciones-electricas-y-automaticas/


## 5. Extraer la información de cada ficha

Visitaremos cada enlace y recogeremos los campos visibles de la ficha del ciclo.

In [8]:
def limpiar_texto(elemento):
    if elemento is None:
        return ""
    return " ".join(elemento.get_text(" ", strip=True).split())


def textos_de_lista(ficha, texto_inicial):
    for parrafo in ficha.select("main p"):
        if limpiar_texto(parrafo).startswith(texto_inicial):
            lista = parrafo.find_next("ul")
            if lista is not None and lista.find_parent("main") is not None:
                return [limpiar_texto(item) for item in lista.find_all("li", recursive=False)]
    return []

ciclos = []

for enlace_ciclo in enlaces_ciclos:
    respuesta_ficha = requests.get(enlace_ciclo["url"], timeout=30)
    respuesta_ficha.raise_for_status()
    ficha = BeautifulSoup(respuesta_ficha.text, "html.parser")

    parrafos = [limpiar_texto(parrafo) for parrafo in ficha.select("main p")]
    campos = {
        "familia_profesional": next((p.replace("Familia profesional:", "").strip() for p in parrafos if p.startswith("Familia profesional:")), ""),
        "duracion": next((p.replace("Duración total:", "").strip() for p in parrafos if p.startswith("Duración total:")), ""),
        "modelo": next((p.replace("Modelo:", "").strip() for p in parrafos if p.startswith("Modelo:")), ""),
        "turno": next((p.replace("Turno:", "").strip() for p in parrafos if p.startswith("Turno:")), ""),
        "metodologia": next((p.replace("Metodología:", "").strip() for p in parrafos if p.startswith("Metodología:")), "")
    }

    ciclos.append({
        "nombre": limpiar_texto(ficha.find("h1")) or enlace_ciclo["nombre"],
        "url": enlace_ciclo["url"],
        **campos,
        "acceso": textos_de_lista(ficha, "El acceso a los ciclos"),
        "salidas_profesionales": textos_de_lista(ficha, "Trabajar en:"),
        "continuidad": textos_de_lista(ficha, "Seguir estudiando:")
    })

print("Fichas extraídas:", len(ciclos))
for ciclo in ciclos:
    print(ciclo["nombre"], "-", ciclo["duracion"])

Fichas extraídas: 2
Gestión administrativa - 2 años - Primer año - Segundo año
Instalaciones eléctricas y automáticas - 2 años - Primer año - Segundo año


## 6. Mostrar un resumen de los datos extraídos

In [9]:
for ciclo in ciclos:
    print("Nombre:", ciclo["nombre"])
    print("Familia profesional:", ciclo["familia_profesional"])
    print("Duración:", ciclo["duracion"])
    print("Modelo:", ciclo["modelo"])
    print("Turno:", ciclo["turno"])
    print("Metodología:", ciclo["metodologia"])
    print("Accesos encontrados:", len(ciclo["acceso"]))
    print("Salidas profesionales encontradas:", len(ciclo["salidas_profesionales"]))
    print("Continuidad encontrada:", len(ciclo["continuidad"]))
    print("-" * 60)

Nombre: Gestión administrativa
Familia profesional: Administrativo
Duración: 2 años - Primer año - Segundo año
Modelo: A y D (Castellano y Euskera)
Turno: D_Matutino (08:00-14:30) y A_vespertino (15:00-21:30)
Metodología: Aprendizaje colaborativo basado en retos
Accesos encontrados: 8
Salidas profesionales encontradas: 10
Continuidad encontrada: 4
------------------------------------------------------------
Nombre: Instalaciones eléctricas y automáticas
Familia profesional: Electricidad
Duración: 2 años - Primer año - Segundo año
Modelo: A (Castellano)
Turno: Matutino (08:00-14:30)
Metodología: 
Accesos encontrados: 8
Salidas profesionales encontradas: 1
Continuidad encontrada: 4
------------------------------------------------------------


## 7. Guardar los resultados en CSV

In [10]:
carpeta_salida = Path.cwd()
ruta_csv = carpeta_salida / "ciclos_tartanga.csv"

campos_csv = [
    "nombre",
    "url",
    "familia_profesional",
    "duracion",
    "modelo",
    "turno",
    "metodologia",
    "acceso",
    "salidas_profesionales",
    "continuidad"
]

filas_csv = []
for ciclo in ciclos:
    fila = ciclo.copy()
    fila["acceso"] = " | ".join(fila["acceso"])
    fila["salidas_profesionales"] = " | ".join(fila["salidas_profesionales"])
    fila["continuidad"] = " | ".join(fila["continuidad"])
    filas_csv.append(fila)

with ruta_csv.open("w", newline="", encoding="utf-8") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=campos_csv)
    escritor.writeheader()
    escritor.writerows(filas_csv)

print("CSV guardado en:", ruta_csv)

CSV guardado en: c:\Users\marke\Desktop\CLASE\API REST\Scrapy\tartanga\ciclos_tartanga.csv


## 8. Guardar los resultados en JSON

In [11]:
ruta_json = carpeta_salida / "ciclos_tartanga.json"

with ruta_json.open("w", encoding="utf-8") as archivo:
    json.dump(ciclos, archivo, ensure_ascii=False, indent=4)

print("JSON guardado en:", ruta_json)

JSON guardado en: c:\Users\marke\Desktop\CLASE\API REST\Scrapy\tartanga\ciclos_tartanga.json


## 9. Comprobar los archivos generados

El CSV contiene una fila por ciclo y el JSON conserva las listas completas de cada ficha.

## 10. Leer el JSON generado

In [12]:
with ruta_json.open("r", encoding="utf-8") as archivo:
    ciclos_guardados = json.load(archivo)

print("Registros leídos del JSON:", len(ciclos_guardados))
for ciclo in ciclos_guardados:
    print(f"{ciclo['nombre']} -> {ciclo['url']}")

Registros leídos del JSON: 2
Gestión administrativa -> https://www.tartanga.eus/formacion/ciclos-formativos-de-grado-medio-grados-d/gestion-administrativa/
Instalaciones eléctricas y automáticas -> https://www.tartanga.eus/formacion/ciclos-formativos-de-grado-medio-grados-d/instalaciones-electricas-y-automaticas/


## 11. Resultado final

In [13]:
print(f"Scraping terminado: {len(ciclos)} ciclos guardados.")
print(f"CSV: {ruta_csv}")
print(f"JSON: {ruta_json}")

Scraping terminado: 2 ciclos guardados.
CSV: c:\Users\marke\Desktop\CLASE\API REST\Scrapy\tartanga\ciclos_tartanga.csv
JSON: c:\Users\marke\Desktop\CLASE\API REST\Scrapy\tartanga\ciclos_tartanga.json
